# 🚧 JalanKita — YOLO26s Training
**Project**: Deteksi Kerusakan Jalan Sidoarjo  
**Classes**: lubang_besar, lubang_kecil, retak_kulit_buaya, retak_memanjang  
**Platform**: Amazon SageMaker Studio (JupyterLab Space)  
**Model**: YOLO26s (NMS-free, MuSGD optimizer, ProgLoss+STAL)

---
> Dataset clean sudah ada di `/home/sagemaker-user/dataset_clean/`

## ⚙️ KONFIGURASI

In [ ]:
from pathlib import Path

EPOCHS      = 200
BATCH_SIZE  = 16
IMG_SIZE    = 640
PATIENCE    = 50
SAVE_PERIOD = 5

PROJECT_NAME = "jalankita"
RUN_NAME     = "yolo26s_v1"

DATASET_CLEAN = Path("/home/sagemaker-user/dataset_clean")
YAML_PATH     = DATASET_CLEAN / "data.yaml"

print("Konfigurasi tersimpan!")
print(f"  Dataset : {YAML_PATH}")
print(f"  Epochs  : {EPOCHS} | Batch: {BATCH_SIZE} | ImgSize: {IMG_SIZE}")

## 📦 STEP 1 — Install Dependencies

In [ ]:
import subprocess, sys

print("[1/2] Installing ultralytics (progress otomatis dari pip)...")
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "tqdm"], check=True)

print("[2/2] Import libraries...")
from tqdm import tqdm
import os, shutil, yaml
from pathlib import Path
from ultralytics import YOLO
import torch

print(f"✅ PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU: {result.stdout.strip()}")

## 💡 STEP 2 — Informasi Model

In [ ]:
print("Model: YOLO26s (pretrained)")
print("  - NMS-free end-to-end detection")
print("  - MuSGD optimizer")
print("  - ProgLoss + STAL untuk small object detection")

## 🚀 STEP 3 — Mulai Training YOLO26s

In [ ]:
if not YAML_PATH.exists():
    print(f"❌ data.yaml tidak ditemukan di {YAML_PATH}")
elif not torch.cuda.is_available():
    print("❌ GPU tidak terdeteksi")
else:
    print(f"Dataset  : {YAML_PATH}")
    print(f"Epochs   : {EPOCHS} | Batch: {BATCH_SIZE}")
    print("-" * 50)
    print("📥 Download model yolo26s.pt (progress otomatis dari Ultralytics)...")

    model = YOLO("yolo26s.pt")

    print("🚀 Training dimulai (progress bar epoch per epoch dari Ultralytics)...")

    results = model.train(
        data        = str(YAML_PATH),
        epochs      = EPOCHS,
        imgsz       = IMG_SIZE,
        batch       = BATCH_SIZE,
        device      = 0,
        patience    = PATIENCE,
        save_period = SAVE_PERIOD,
        project     = f"/home/sagemaker-user/runs/{PROJECT_NAME}",
        name        = RUN_NAME,
        exist_ok    = True,
    )

    print("\n" + "="*50)
    print("TRAINING SELESAI!")
    print(f"   mAP50    : {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
    print(f"   mAP50-95 : {results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

## 💾 STEP 4 — Backup Hasil Training

In [ ]:
from datetime import datetime

RUNS_DIR   = Path(f"/home/sagemaker-user/runs/{PROJECT_NAME}/{RUN_NAME}")
BACKUP_DIR = Path("/home/sagemaker-user/backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

timestamp   = datetime.now().strftime("%Y%m%d_%H%M")
backup_name = f"jalankita_{RUN_NAME}_{timestamp}"
backup_path = BACKUP_DIR / backup_name

if RUNS_DIR.exists():
    shutil.make_archive(str(backup_path), "zip", str(RUNS_DIR))
    zip_size = (Path(str(backup_path) + ".zip")).stat().st_size / 1e6
    print(f"✅ Backup: {backup_path}.zip ({zip_size:.1f} MB)")

    weights_dir = RUNS_DIR / "weights"
    for pt_name in ["last.pt", "best.pt"]:
        src = weights_dir / pt_name
        if src.exists():
            shutil.copy(src, Path("/home/sagemaker-user") / pt_name)
else:
    print("❌ Folder runs tidak ditemukan.")

## 📊 STEP 5 — Evaluasi Model (Opsional)

In [ ]:
best_pt = Path("/home/sagemaker-user/best.pt")
if not best_pt.exists():
    best_pt = RUNS_DIR / "weights" / "best.pt"

if best_pt.exists():
    model_eval = YOLO(str(best_pt))
    metrics    = model_eval.val(data=str(YAML_PATH), device=0)

    print("\n📈 Evaluasi per Kelas:")
    class_names = ["lubang_besar", "lubang_kecil", "retak_kulit_buaya", "retak_memanjang"]
    for i, name in enumerate(class_names):
        try:
            ap = metrics.box.ap50[i]
            print(f"  {name:25s}: AP50 = {ap:.4f}")
        except:
            pass
else:
    print("best.pt belum ada.")